In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from project_config import TICKERS, ROLL_WINDOW, Z_THRESHOLD, MAD_SCALE, ANNUALISATION, rolling_mad

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True


## 1. Load data and rebuild the anomaly flags

Same self-contained pattern as the other notebooks.

In [ ]:
DATA_DIR = Path("data")

dfs, returns = {}, {}
for key in TICKERS:
    dfs[key] = pd.read_csv(DATA_DIR / f"{key}.csv", index_col="date", parse_dates=True)
    returns[key] = dfs[key]["Close"].pct_change().dropna()

z_flags = {}
for key in TICKERS:
    r = returns[key]
    roll_median = r.rolling(ROLL_WINDOW).median()
    roll_mad = rolling_mad(r, ROLL_WINDOW) * MAD_SCALE
    z_robust = (r - roll_median) / roll_mad.replace(0, np.nan)
    z_flags[key] = r[z_robust.abs() > Z_THRESHOLD]
    print(f"{TICKERS[key]}: {len(z_flags[key])} flagged days available as potential signals")


## 2. Backtest engine

Loop-based rather than fully vectorised — the dataset is small enough (~2,900 rows) that
performance isn't a concern, and a loop makes the non-overlap and holding-period logic much
easier to verify by reading it, which matters more than speed here.

In [ ]:
HOLD_DAYS = 5
TXN_COST_BPS = 5  # per side — 10bps round-trip per trade, a simple realism check


def backtest_fade_signal(returns_series, flagged_dates, hold_days=HOLD_DAYS, txn_cost_bps=TXN_COST_BPS):
    dates = returns_series.index
    strategy_returns = pd.Series(0.0, index=dates)
    trade_log = []

    position_end_date = None
    for flag_date in sorted(flagged_dates):
        # Skip if still inside a previous trade's holding window — no overlapping positions
        if position_end_date is not None and flag_date <= position_end_date:
            continue

        flag_return = returns_series.loc[flag_date]
        direction = -1 if flag_return > 0 else 1  # fade the move

        flag_loc = dates.get_loc(flag_date)
        entry_loc = flag_loc + 1
        if entry_loc >= len(dates):
            continue  # flagged on the last available day — nothing to trade

        exit_loc = min(entry_loc + hold_days - 1, len(dates) - 1)
        hold_dates = dates[entry_loc:exit_loc + 1]

        daily_pnl = direction * returns_series.loc[hold_dates]
        strategy_returns.loc[hold_dates] += daily_pnl

        round_trip_cost = 2 * (txn_cost_bps / 10_000)
        strategy_returns.loc[hold_dates[0]] -= round_trip_cost

        position_end_date = hold_dates[-1]
        trade_log.append({
            "flag_date": flag_date,
            "direction": "Long" if direction == 1 else "Short",
            "entry_date": hold_dates[0],
            "exit_date": hold_dates[-1],
            "trade_pnl": daily_pnl.sum() - round_trip_cost,
        })

    return strategy_returns, pd.DataFrame(trade_log)


def performance_metrics(daily_returns):
    total_return = (1 + daily_returns).prod() - 1
    n_years = len(daily_returns) / 252
    annualized_return = (1 + total_return) ** (1 / n_years) - 1 if total_return > -1 else np.nan
    sharpe = (daily_returns.mean() / daily_returns.std() * np.sqrt(252)) if daily_returns.std() > 0 else np.nan

    equity_curve = (1 + daily_returns).cumprod()
    drawdown = (equity_curve - equity_curve.cummax()) / equity_curve.cummax()

    return {
        "total_return": total_return,
        "annualized_return": annualized_return,
        "sharpe": sharpe,
        "max_drawdown": drawdown.min(),
    }, equity_curve


## 3. Run it — strategy vs. buy-and-hold, per commodity

In [ ]:
results = {}
for key in TICKERS:
    strategy_returns, trade_log = backtest_fade_signal(returns[key], z_flags[key].index)
    strategy_metrics, strategy_equity = performance_metrics(strategy_returns)
    bh_metrics, bh_equity = performance_metrics(returns[key])

    results[key] = {
        "trade_log": trade_log,
        "strategy_returns": strategy_returns,
        "strategy_equity": strategy_equity,
        "bh_equity": bh_equity,
        "strategy_metrics": strategy_metrics,
        "bh_metrics": bh_metrics,
    }

    n_trades = len(trade_log)
    win_rate = (trade_log["trade_pnl"] > 0).mean() if n_trades else np.nan
    print(f"{TICKERS[key]}: {n_trades} non-overlapping trades, win rate = {win_rate:.0%}" if n_trades else f"{TICKERS[key]}: no trades")


In [ ]:
summary = []
for key, label in TICKERS.items():
    sm, bm = results[key]["strategy_metrics"], results[key]["bh_metrics"]
    summary.append({
        "commodity": label,
        "n_trades": len(results[key]["trade_log"]),
        "strategy_total_return": sm["total_return"],
        "strategy_sharpe": sm["sharpe"],
        "strategy_max_dd": sm["max_drawdown"],
        "buy_hold_total_return": bm["total_return"],
        "buy_hold_sharpe": bm["sharpe"],
        "buy_hold_max_dd": bm["max_drawdown"],
    })

pd.DataFrame(summary).set_index("commodity")


## 4. Equity curves

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
for ax, (key, label) in zip(axes, TICKERS.items()):
    ax.plot(results[key]["bh_equity"].index, results[key]["bh_equity"], label="Buy & hold", linewidth=1, color="steelblue")
    ax.plot(results[key]["strategy_equity"].index, results[key]["strategy_equity"], label="Fade-the-anomaly strategy", linewidth=1, color="darkorange")
    ax.set_title(label)
    ax.set_ylabel("Growth of $1")
    ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()


## 5. Trade log (for spot-checking)

In [ ]:
for key, label in TICKERS.items():
    print(f"\n=== {label} ===")
    display(results[key]["trade_log"])
